# Revue audio manuelle — Waxal + Common Voice

Ce notebook charge les résultats de l'audit, construit une file priorisée et permet d'écouter les exemples avant la préparation du mélange d'entraînement. Chaque décision est sauvegardée immédiatement.

Décisions : `keep` (conserver), `trim` (rogner), `segment` (segmenter avec alignement) et `exclude` (exclure). L'écoute des splits de validation et de test sert uniquement à vérifier leur intégrité, jamais à choisir ou régler le modèle.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets[audio]>=4.0,<5", "numpy>=1.26,<3",
    "soundfile>=0.12,<1", "pandas>=2,<3", "ipywidgets>=8,<9",
])
print("Dépendances de revue installées.")

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/ngangoula-jaures/Finetune_omniASR_CTC_1B_for_baoule.git"
PROJECT_DIR = Path("/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule")

if not PROJECT_DIR.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)])
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"])

required = PROJECT_DIR / "src" / "create_review_queue.py"
if not required.is_file():
    raise FileNotFoundError("Le script de revue est absent. Pousse d'abord les changements vers GitHub.")
print("Projet :", PROJECT_DIR)

## Retrouver l'audit

Le notebook utilise d'abord l'audit présent dans `/kaggle/working`, puis les deux fichiers minimaux versionnés avec le dépôt. Il fonctionne ainsi dans une nouvelle session Kaggle sans ajouter l'audit comme Dataset d'entrée. `AUDIT_DIR_OVERRIDE` reste disponible pour sélectionner manuellement un autre audit.

In [ ]:
AUDIT_DIR_OVERRIDE = None  # Exemple : Path('/kaggle/input/mon-audit/baoule_ctc1b_dataset_audit')

def is_audit_dir(path):
    path = Path(path)
    return (path / "dataset_audit.csv").is_file() and (path / "duplicate_groups.json").is_file()

if AUDIT_DIR_OVERRIDE is not None:
    AUDIT_DIR = Path(AUDIT_DIR_OVERRIDE)
elif is_audit_dir('/kaggle/working/baoule_ctc1b_dataset_audit'):
    AUDIT_DIR = Path('/kaggle/working/baoule_ctc1b_dataset_audit')
elif is_audit_dir(PROJECT_DIR / 'baoule_ctc1b_dataset_audit'):
    AUDIT_DIR = PROJECT_DIR / 'baoule_ctc1b_dataset_audit'
else:
    candidates = sorted({
        path.parent
        for root in (Path('/kaggle/input'), Path('/kaggle/working'))
        if root.exists()
        for path in root.rglob('dataset_audit.csv')
        if is_audit_dir(path.parent) and 'baoule_ctc1b_review' not in path.parts
    })
    if len(candidates) != 1:
        listing = '\n'.join(map(str, candidates)) or '(aucun)'
        raise RuntimeError(f"Audit non déterminé. Candidats :\n{listing}\nRenseigne AUDIT_DIR_OVERRIDE.")
    AUDIT_DIR = candidates[0]

if not is_audit_dir(AUDIT_DIR):
    raise FileNotFoundError(f"Audit incomplet : {AUDIT_DIR}")
print("Audit :", AUDIT_DIR)

In [ ]:
import json
import os
import shutil

REVIEW_DIR = Path('/kaggle/working/baoule_ctc1b_review')
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
BASE_MANIFEST = REVIEW_DIR / 'decision_manifest.csv'
REVIEWED_MANIFEST = REVIEW_DIR / 'decision_manifest_reviewed.csv'
PREVIOUS_REVIEW_MANIFEST = None  # Chemin facultatif vers une sauvegarde précédente

if PREVIOUS_REVIEW_MANIFEST is not None:
    previous = Path(PREVIOUS_REVIEW_MANIFEST)
    if not previous.is_file():
        raise FileNotFoundError(previous)
    shutil.copy2(previous, REVIEWED_MANIFEST)
elif not REVIEWED_MANIFEST.exists():
    previous_candidates = sorted(Path('/kaggle/input').rglob('decision_manifest_reviewed.csv'))
    if len(previous_candidates) == 1:
        shutil.copy2(previous_candidates[0], REVIEWED_MANIFEST)
        print('Progression précédente restaurée :', previous_candidates[0])
    elif len(previous_candidates) > 1:
        print('Plusieurs sauvegardes détectées ; renseigne PREVIOUS_REVIEW_MANIFEST :', previous_candidates)

if not BASE_MANIFEST.exists():
    command = [
        sys.executable, str(PROJECT_DIR / 'src' / 'create_review_queue.py'),
        '--audit-csv', str(AUDIT_DIR / 'dataset_audit.csv'),
        '--duplicates-json', str(AUDIT_DIR / 'duplicate_groups.json'),
        '--output-dir', str(REVIEW_DIR),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True, env={**os.environ})
else:
    print('Manifeste existant conservé :', BASE_MANIFEST)

print((REVIEW_DIR / 'review_queue_summary.json').read_text(encoding='utf-8'))

## Charger les audios aux révisions auditées

Le cache de l'audit est réutilisé quand il existe. Les audios restent dans leur format compressé pour une lecture plus légère, sans conversion préalable.

In [ ]:
from datasets import Audio, load_dataset

sys.path.insert(0, str(PROJECT_DIR))
from src.audit_datasets import DEFAULT_KLAYT_REVISION, DEFAULT_WAXAL_REVISION

CACHE_DIR = '/kaggle/working/hf-cache'
waxal = load_dataset('google/WaxalNLP', 'bau_tts', revision=DEFAULT_WAXAL_REVISION, cache_dir=CACHE_DIR)
klayt = load_dataset('Klayt/baoule-common-voice', revision=DEFAULT_KLAYT_REVISION, cache_dir=CACHE_DIR)
DATASETS = {
    'waxal': {name: split.cast_column('audio', Audio(decode=False)) for name, split in waxal.items()},
    'klayt': {name: split.cast_column('audio', Audio(decode=False)) for name, split in klayt.items()},
}
print('Datasets prêts aux révisions auditées.')

## Interface de revue

Pour `trim`, vérifie et ajuste le début et la fin proposés. Pour `segment`, les frontières sont facultatives à ce stade : le découpage final exigera ensuite un alignement audio–texte. Utilise les notes pour signaler une voix parasite, une mauvaise transcription, du bruit ou toute ambiguïté.

In [ ]:
import base64
import html
import json
import mimetypes
import os
import tempfile

import ipywidgets as widgets
import pandas as pd
from IPython.display import Audio as IPythonAudio, HTML, display

ALLOWED = {'', 'keep', 'trim', 'segment', 'exclude'}
manifest_path = REVIEWED_MANIFEST if REVIEWED_MANIFEST.exists() else BASE_MANIFEST
manifest = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
base_ids = set(pd.read_csv(BASE_MANIFEST, usecols=['review_id'], dtype=str)['review_id'])
if set(manifest['review_id']) != base_ids:
    raise ValueError('La sauvegarde de revue ne correspond pas exactement à cet audit.')
required_review_columns = {
    'rms_dbfs', 'leading_silence_seconds', 'trailing_silence_seconds',
    'text_raw', 'segment_boundaries_seconds',
}
if missing_columns := required_review_columns.difference(manifest.columns):
    raise ValueError(f'Manifeste ancien ou incomplet ; champs absents : {sorted(missing_columns)}')
missing = ALLOWED.difference(set(manifest['final_decision'].unique()))
invalid = set(manifest['final_decision'].unique()).difference(ALLOWED)
if invalid:
    raise ValueError(f'Décisions invalides dans le manifeste : {sorted(invalid)}')

review_order = (
    manifest.loc[manifest['suggested_decision'] != 'keep']
    .assign(
        _priority=lambda frame: pd.to_numeric(frame['priority']),
        _row_index=lambda frame: pd.to_numeric(frame['row_index']),
    )
    .sort_values(['_priority', 'source', 'split', '_row_index'])
    .index.tolist()
)
if not review_order:
    raise RuntimeError('La file de revue est vide.')
current = [review_order[0]]

def atomic_save_dataframe(frame, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, destination)

def save_progress():
    atomic_save_dataframe(manifest, REVIEWED_MANIFEST)
    counts = manifest['final_decision'].replace('', 'pending').value_counts().to_dict()
    progress = {
        'total_examples': int(len(manifest)),
        'review_queue_examples': int(len(review_order)),
        'reviewed_queue_examples': int(sum(bool(manifest.at[index, 'final_decision']) for index in review_order)),
        'pending_queue_examples': int(sum(not bool(manifest.at[index, 'final_decision']) for index in review_order)),
        'decision_counts_all_examples': {str(key): int(value) for key, value in counts.items()},
    }
    (REVIEW_DIR / 'review_progress.json').write_text(
        json.dumps(progress, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return progress

def original_audio_player(row):
    item = DATASETS[row['source']][row['split']][int(row['row_index'])]
    audio = item['audio']
    if audio.get('bytes') is not None:
        mime = mimetypes.guess_type(str(audio.get('path') or 'audio.wav'))[0] or 'audio/wav'
        encoded = base64.b64encode(audio['bytes']).decode('ascii')
        return HTML(f'<audio controls preload="metadata" style="width:100%" src="data:{mime};base64,{encoded}"></audio>')
    if audio.get('path'):
        return IPythonAudio(filename=audio['path'])
    raise ValueError('Audio sans bytes ni chemin.')

output = widgets.Output()
status = widgets.HTML()
notes = widgets.Textarea(description='Notes', layout=widgets.Layout(width='100%', height='90px'))
trim_start = widgets.FloatText(description='Début (s)', value=0.0)
trim_end = widgets.FloatText(description='Fin (s)', value=0.0)
boundaries = widgets.Text(description='Coupes (s)', placeholder='ex. 24.8, 52.1', layout=widgets.Layout(width='60%'))
source_filter = widgets.Dropdown(description='Source', options=['toutes', 'waxal', 'klayt'], value='toutes')
split_filter = widgets.Dropdown(description='Split', options=['tous', 'train', 'validation', 'test'], value='tous')

def filtered_order():
    result = []
    for index in review_order:
        row = manifest.loc[index]
        if source_filter.value != 'toutes' and row['source'] != source_filter.value:
            continue
        if split_filter.value != 'tous' and row['split'] != split_filter.value:
            continue
        result.append(index)
    return result

def set_default_bounds(row):
    duration = float(row['duration_seconds'])
    leading = float(row['leading_silence_seconds'] or 0.0)
    trailing = float(row['trailing_silence_seconds'] or 0.0)
    trim_start.value = round(max(0.0, leading), 3)
    trim_end.value = round(max(trim_start.value, duration - trailing), 3)

def render(index=None):
    order = filtered_order()
    if not order:
        status.value = '<b>Aucun exemple pour ce filtre.</b>'
        return
    if index not in order:
        pending = [item for item in order if not manifest.at[item, 'final_decision']]
        index = pending[0] if pending else order[0]
    current[0] = index
    row = manifest.loc[index]
    notes.value = row['review_notes']
    boundaries.value = row['segment_boundaries_seconds']
    if row['trim_start_seconds'] and row['trim_end_seconds']:
        trim_start.value = float(row['trim_start_seconds'])
        trim_end.value = float(row['trim_end_seconds'])
    else:
        set_default_bounds(row)
    progress = save_progress()
    position = order.index(index) + 1
    status.value = (
        f'<b>{position}/{len(order)}</b> avec ce filtre — '
        f"{progress['reviewed_queue_examples']}/{progress['review_queue_examples']} décisions enregistrées — "
        f"{progress['pending_queue_examples']} restantes"
    )
    metadata = {
        'review_id': row['review_id'], 'priorité': row['priority'],
        'durée (s)': row['duration_seconds'], 'mots/s': row['words_per_second'],
        'RMS (dBFS)': row['rms_dbfs'],
        'silence début/fin (s)': f"{row['leading_silence_seconds']} / {row['trailing_silence_seconds']}",
        'raisons audit': row['review_reasons'] or 'aucune',
        'suggestion': f"{row['suggested_decision']} — {row['suggestion_rationale']}",
        'décision actuelle': row['final_decision'] or 'À FAIRE',
    }
    table = ''.join(
        f'<tr><th style="text-align:left;padding-right:12px">{html.escape(str(key))}</th>'
        f'<td>{html.escape(str(value))}</td></tr>' for key, value in metadata.items()
    )
    with output:
        output.clear_output(wait=True)
        display(HTML(f'<table>{table}</table>'))
        display(HTML('<p><b>Texte brut</b><br>' + html.escape(row['text_raw']) + '</p>'))
        display(HTML('<p><b>Texte normalisé</b><br>' + html.escape(row['text_normalized']) + '</p>'))
        try:
            display(original_audio_player(row))
        except Exception as error:
            display(HTML(f'<b>Lecture impossible :</b> {html.escape(str(error))}'))

def parse_boundaries(value, duration):
    if not value.strip():
        return ''
    points = [float(part.strip()) for part in value.split(',') if part.strip()]
    if points != sorted(set(points)) or any(point <= 0 or point >= duration for point in points):
        raise ValueError('Les coupes doivent être uniques, croissantes et comprises dans la durée.')
    return ','.join(f'{point:.3f}' for point in points)

def record(decision):
    if decision not in ALLOWED:
        raise ValueError(decision)
    index = current[0]
    duration = float(manifest.at[index, 'duration_seconds'])
    if decision == 'trim' and not (0.0 <= trim_start.value < trim_end.value <= duration + 1e-3):
        raise ValueError(f'Bornes trim invalides pour une durée de {duration:.3f} s.')
    manifest.at[index, 'final_decision'] = decision
    manifest.at[index, 'review_notes'] = notes.value.strip()
    manifest.at[index, 'trim_start_seconds'] = f'{trim_start.value:.3f}' if decision == 'trim' else ''
    manifest.at[index, 'trim_end_seconds'] = f'{trim_end.value:.3f}' if decision == 'trim' else ''
    manifest.at[index, 'segment_boundaries_seconds'] = parse_boundaries(boundaries.value, duration) if decision == 'segment' else ''
    save_progress()
    order = filtered_order()
    start = order.index(index) + 1
    next_pending = next((item for item in order[start:] + order[:start] if not manifest.at[item, 'final_decision']), index)
    render(next_pending)

def move(delta):
    order = filtered_order()
    if not order:
        return
    position = order.index(current[0]) if current[0] in order else 0
    render(order[(position + delta) % len(order)])

buttons = []
for label, decision, style in [
    ('Conserver', 'keep', 'success'), ('Rogner', 'trim', 'warning'),
    ('Segmenter', 'segment', 'info'), ('Exclure', 'exclude', 'danger'),
    ('Remettre à faire', '', ''),
]:
    button = widgets.Button(description=label, button_style=style)
    def handle_click(_, value=decision):
        try:
            record(value)
        except Exception as error:
            status.value = f'<b style="color:#c00">Décision non enregistrée : {html.escape(str(error))}</b>'
    button.on_click(handle_click)
    buttons.append(button)
previous_button = widgets.Button(description='← Précédent')
next_button = widgets.Button(description='Suivant →')
previous_button.on_click(lambda _: move(-1))
next_button.on_click(lambda _: move(1))
source_filter.observe(lambda change: render(None) if change['name'] == 'value' else None)
split_filter.observe(lambda change: render(None) if change['name'] == 'value' else None)

display(widgets.VBox([
    widgets.HBox([source_filter, split_filter]), status, output, notes,
    widgets.HBox([trim_start, trim_end]), boundaries,
    widgets.HBox(buttons), widgets.HBox([previous_button, next_button]),
]))
render(current[0])

## Exporter la progression

Exécute cette cellule avant de quitter Kaggle, puis télécharge le ZIP. Pour reprendre plus tard, ajoute le dossier de revue précédent comme entrée Kaggle et copie son fichier `decision_manifest_reviewed.csv` dans `REVIEW_DIR` avant d'exécuter la cellule d'interface.

In [ ]:
import shutil

save_progress()
archive = shutil.make_archive('/kaggle/working/baoule_ctc1b_review', 'zip', REVIEW_DIR)
print('Archive prête à télécharger :', archive)
print((REVIEW_DIR / 'review_progress.json').read_text(encoding='utf-8'))